In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ensure models directory exists
os.makedirs('../models', exist_ok=True)

# ==========================================
# PART 1: LOAD DATA & PREPARE FOR MODELING
# ==========================================
print("Loading processed dataset...")
# Load the upgraded dataset generated in Phase 2
data_path = '../data/processed/full_network_ml_upgraded.csv'
df = pd.read_csv(data_path)

# 1. Define Features (X) and Target (y)
target = 'delay_minutes'
# We drop the target column to get our feature matrix X
X = df.drop(columns=[target])
y = df[target]

print(f"Dataset loaded. Total records: {len(df)}")
print(f"Features included: {list(X.columns)}")

# 2. Train-Test Split (80% Training, 20% Testing)
# random_state=42 ensures we get the exact same split every time we run this
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining data shape (X): {X_train.shape}")
print(f"Testing data shape (X): {X_test.shape}")

# 3. Feature Scaling
# Standardize features by removing the mean and scaling to unit variance
scaler = StandardScaler()

# We only FIT the scaler on the training data to prevent "data leakage"
# Then we transform both train and test sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for use in the final app/dashboard
joblib.dump(scaler, '../models/phase3_feature_scaler.pkl')
print("\nScaler saved to '../models/phase3_feature_scaler.pkl'")
print("PART 1 Complete: Data is ready for modeling!")

Loading processed dataset...
Dataset loaded. Total records: 61037
Features included: ['station_id_encoded', 'hour', 'is_peak', 'station_congestion', 'stop_sequence', 'prev_delay', 'route_encoded', 'day_of_week', 'is_weekend']

Training data shape (X): (48829, 9)
Testing data shape (X): (12208, 9)

Scaler saved to '../models/phase3_feature_scaler.pkl'
PART 1 Complete: Data is ready for modeling!


In [2]:
# ==========================================
# PART 2: BASELINE MODELS
# ==========================================
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Initializing Baseline Models...")

# We will store our evaluation metrics in this list to build a comparison table later
model_results = []

# Helper function to evaluate and store metrics cleanly
def evaluate_and_store(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    model_results.append({
        'Model': model_name,
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'R2 Score': round(r2, 4)
    })
    print(f"{model_name} Evaluated -> RMSE: {rmse:.4f} mins | MAE: {mae:.4f} mins | R2: {r2:.4f}")

# 1. Train Linear Regression
print("\nTraining Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
evaluate_and_store('Linear Regression', y_test, lr_preds)

# 2. Train Decision Tree Regressor
print("Training Decision Tree Regressor...")
dt_model = DecisionTreeRegressor(random_state=42)
# Note: Decision Trees don't technically need scaled data, but using X_train_scaled 
# is perfectly fine and keeps our pipeline consistent.
dt_model.fit(X_train_scaled, y_train)
dt_preds = dt_model.predict(X_test_scaled)
evaluate_and_store('Decision Tree Regressor', y_test, dt_preds)

# Display the comparison table for our baselines
baseline_results_df = pd.DataFrame(model_results)
print("\n--- Baseline Models Comparison ---")
print(baseline_results_df.to_string(index=False))

Initializing Baseline Models...

Training Linear Regression...
Linear Regression Evaluated -> RMSE: 1.3252 mins | MAE: 0.9768 mins | R2: 0.9200
Training Decision Tree Regressor...
Decision Tree Regressor Evaluated -> RMSE: 1.6469 mins | MAE: 1.2139 mins | R2: 0.8765

--- Baseline Models Comparison ---
                  Model   RMSE    MAE  R2 Score
      Linear Regression 1.3252 0.9768    0.9200
Decision Tree Regressor 1.6469 1.2139    0.8765
